# Task 1 — Equity research pipeline

Prices → indicators (no TA-Lib) → news → LLM signal → HTML brief.

**Env:** local uses `.env`. Colab uses Secrets (key icon). `src/config.py` selects the source.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# After you create the GitHub repo, put the HTTPS clone URL here.
REPO_URL = os.environ.get(
    "FINANCIAL_AI_REPO",
    "https://github.com/lavanblavan/Financial_AI.git",
)

def ensure_project() -> Path:
    if Path("src/config.py").exists():
        root = Path.cwd().resolve()
    elif IN_COLAB:
        root = Path("/content/Financial_AI")
        if not (root / "src/config.py").exists():
            subprocess.run(["git", "clone", REPO_URL, str(root)], check=True)
        os.chdir(root)
    else:
        raise FileNotFoundError("Run this notebook from the repo root so src/ is visible.")

    req = root / "requirements.txt"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=True)
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))
    return root

ROOT = ensure_project()
print("project root:", ROOT)
print("runtime:", "colab" if IN_COLAB else "local")

In [ ]:
from src.config import describe_env, load_settings

settings = load_settings()
print(describe_env(settings))
if not settings.llm_ready:
    print(
        "LLM key missing. Local: copy .env.example to .env and set GROQ_API_KEY. "
        "Colab: add GROQ_API_KEY in Secrets and enable notebook access."
    )

In [ ]:
from src.data import fetch_prices

prices = fetch_prices(settings.ticker, period=settings.lookback)
prices.tail()

In [ ]:
from src.indicators import add_indicators, latest_snapshot

df = add_indicators(prices)
snapshot = latest_snapshot(df)
display(df[["Close", "sma_20", "sma_50", "rsi_14", "macd", "macd_hist"]].tail())
snapshot

In [ ]:
from src.news import fetch_news

headlines = fetch_news(settings.ticker, min_items=10)
print(f"{len(headlines)} headlines")
headlines

In [ ]:
from src.signal import LLMNotConfiguredError, build_signal

try:
    result = build_signal(snapshot, headlines, settings, settings.ticker)
except LLMNotConfiguredError as exc:
    result = {
        "signal": "HOLD",
        "confidence": 0.0,
        "horizon": "n/a",
        "rationale": str(exc),
        "bull_case": "",
        "bear_case": "",
        "risks": ["LLM key not configured"],
    }
result

In [ ]:
from IPython.display import HTML, display
from src.report import render_brief

brief_path = render_brief(settings.ticker, df, snapshot, headlines, result)
print("wrote", brief_path)
display(HTML(brief_path.read_text(encoding="utf-8")))